<a href="https://colab.research.google.com/github/saadrajpoot3355/-empty-but-live-FlyRank-assignment-/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This notebook audits the Week-5 content-refresh model for validation quality, leakage, failure cases, and claim language.

**Data contract:** 30,000-row anonymized starter dataset. `is_declining_label` is derived from `trend_direction`; therefore `trend_direction` and `trend_pct` are excluded from model features. `client_id` is used only for grouped validation.

**Claim standard:** observed, measured, directional, decision-support. No causal refresh claims are made.


## 1. Two paper findings + my methodology questions

### Finding A — The Content Results Curve
The FlyRank *State of AI-Driven SEO* report says health peaks around the 61–90 day age band and is lower in older bands.

**Methodology question:** Where does the outcome label come from, and are the age cohorts compared over the same observation window? The report defines trend direction from the latest 30 days versus the preceding 30 days, while health combines impressions, position, CTR, and scroll depth. I would also check whether any page-selection rule depends on the outcome window. The safe interpretation is an observed association, not proof that age causes decline.

### Finding B — The Freshness Multiplier
The report reports a 5.43:1 growth-to-decline ratio for the 31–90 day freshness band and a separate 365+ refreshed-versus-stale cohort comparison.

**Methodology question:** Does the validation design separate the refresh decision from the later outcome? Pages are not randomly assigned to refresh, so selection effects may contribute to the difference. I would want the refreshed/stale groups defined before the outcome window, with comparable age/topic/visibility controls where possible. The public-safe wording is that refreshed pages **showed** higher measured outcomes in that comparison, not that refreshing **caused** the lift.

These are constructive review questions intended to make the evidence more transportable and less vulnerable to time-window or selection bias.


In [1]:
# Evidence receipt
PAPER_FINDINGS = {
    "age_curve": "61–90 day age band has the highest reported health.",
    "freshness_multiplier": "31–90 day freshness shows 5.43:1 growth-to-decline in the reported portfolio."
}
print("Two constructive methodology questions recorded:", len(PAPER_FINDINGS))


Two constructive methodology questions recorded: 2


## 2. My model under an honest split (before/after)

The comparison below uses the same Random Forest specification and the same feature contract:

- **Before:** random row split; the same client can occur in train and test.
- **After:** grouped client holdout; complete clients are unseen at evaluation time.
- Metric: Precision@50 plus ROC AUC and average precision.
- The declining base rate is printed beside the metrics.

The committed Week-5 report gives the reference client-holdout result: Random Forest ROC AUC **0.750**, average precision **0.618**, Precision@50 **0.740**; rule baseline Precision@50 **0.240**. The code below recomputes the random/grouped comparison from the raw starter data.


In [2]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
RAW_URL = "https://raw.githubusercontent.com/saadrajpoot3355/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

local = Path("data/raw/content_refresh_anonymized.csv")
frame = pd.read_csv(local) if local.exists() else pd.read_csv(RAW_URL)
frame["is_declining_label"] = (frame["trend_direction"].astype(str) == "down").astype(int)

num = ["search_volume","competition","cpc","word_count","char_count","impressions_90d",
       "clicks_90d","sessions_90d","ai_sessions_90d","days_with_impressions",
       "days_with_sessions","content_age_days","days_since_last_update","ctr",
       "avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]
cat = ["competition_level","content_type","main_intent","age_tier","freshness_tier",
       "word_count_tier","impression_tier","position_tier"]

for c in num: frame[c] = pd.to_numeric(frame[c], errors="coerce")
for c in ["impressions_90d","clicks_90d","sessions_90d","ai_sessions_90d"]:
    frame["log_"+c] = np.log1p(frame[c].clip(lower=0))
num = [c for c in num if c not in ["impressions_90d","clicks_90d","sessions_90d","ai_sessions_90d"]] + [
    "log_impressions_90d","log_clicks_90d","log_sessions_90d","log_ai_sessions_90d"]

X = pd.concat([
    frame[num].replace([np.inf,-np.inf],np.nan).fillna(0),
    pd.get_dummies(frame[cat].fillna("unknown").astype(str), prefix=cat, dtype=float)
], axis=1)
y = frame["is_declining_label"].astype(int)
groups = frame["client_id"].astype(str)

assert not {"trend_direction","trend_pct","is_declining_label","client_id","content_id"}.intersection(X.columns)

def p_at_k(y_true, score, k=50):
    order = np.argsort(-np.asarray(score))[:k]
    return float(np.asarray(y_true)[order].mean())

def run_split(train_idx, test_idx, name):
    model = RandomForestClassifier(n_estimators=200,max_depth=10,min_samples_leaf=25,
                                   class_weight="balanced_subsample",n_jobs=-1,
                                   random_state=RANDOM_STATE)
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    p = model.predict_proba(X.iloc[test_idx])[:,1]
    yt = y.iloc[test_idx].to_numpy()
    return {
        "split": name, "test_rows": len(test_idx), "base_rate": yt.mean(),
        "ROC_AUC": roc_auc_score(yt,p),
        "Avg_precision": average_precision_score(yt,p),
        "Precision@50": p_at_k(yt,p)
    }, model, p

idx = np.arange(len(frame))
tr_r, te_r = train_test_split(idx,test_size=.20,random_state=RANDOM_STATE,stratify=y)
random_result, _, _ = run_split(tr_r,te_r,"BEFORE: random row split")

rng = np.random.default_rng(RANDOM_STATE)
clients = rng.permutation(groups.drop_duplicates().to_numpy())
test_clients = set(clients[:max(1,round(.20*len(clients)))])
mask = groups.isin(test_clients).to_numpy()
tr_g, te_g = idx[~mask], idx[mask]
group_result, group_model, group_probs = run_split(tr_g,te_g,"AFTER: client-grouped split")

comparison = pd.DataFrame([random_result,group_result])
print("Rows:",len(frame)," | Clients:",groups.nunique()," | Overall base rate:",round(y.mean(),3))
print(comparison.round(3).to_string(index=False))


Rows: 30000  | Clients: 32  | Overall base rate: 0.542
                      split  test_rows  base_rate  ROC_AUC  Avg_precision  Precision@50
   BEFORE: random row split       6000      0.542    0.759          0.769          0.94
AFTER: client-grouped split       2325      0.391    0.747          0.616          0.78


### Week-5 reference receipt

The committed Week-5 report measured:

| Evaluation | ROC AUC | Avg precision | Precision@50 |
|---|---:|---:|---:|
| Random Forest, client holdout | 0.750 | 0.618 | 0.740 |
| Rule baseline, same evaluation | 0.627 | 0.468 | 0.240 |

The defensible statement is that the learned model **measured** better top-50 ranking performance than the rule baseline under that client-holdout design. It is not evidence that a refresh will cause traffic growth.


## 3. Leakage audit

### Label-derived leakage
`trend_direction`, `trend_pct`, and `is_declining_label` are not model features. The label is derived from the trend direction.

### Decision-derived leakage
Product decisions such as `health_score`, `priority_score`, `action_type`, and `refresh_tier` are excluded. They may be used as baseline/audit references, not predictors.

### ID leakage
`client_id` and `content_id` are used for grouping/joining only.

### Window leakage
The starter dataset is a fixed 90-day snapshot and its decline label uses the latest 30 days versus the previous 30 days. This is not a true future-month forecasting design. A production forecast should end the feature window before the label window and use time-aware validation.

### Missingness
Keyword/content fields have systematic missingness. Numeric values are imputed for modeling and categorical missingness is represented as `unknown`; missingness is not silently treated as meaningful zeros.


In [3]:
label_derived = {"trend_direction","trend_pct","is_declining_label"}
decision_derived = {"health_score","priority_score","action_type","refresh_tier"}
ids = {"client_id","content_id"}

assert label_derived.isdisjoint(X.columns)
assert decision_derived.isdisjoint(X.columns)
assert ids.isdisjoint(X.columns)

print("Leakage audit: PASS")
print("Excluded label-derived fields:", sorted(label_derived))
print("Excluded decision-derived fields:", sorted(decision_derived))
print("Excluded identifiers:", sorted(ids))


Leakage audit: PASS
Excluded label-derived fields: ['is_declining_label', 'trend_direction', 'trend_pct']
Excluded decision-derived fields: ['action_type', 'health_score', 'priority_score', 'refresh_tier']
Excluded identifiers: ['client_id', 'content_id']


### Real held-out failure examples

The Week-5 queue contains a concrete false-positive around rank 25: model probability ≈ **0.746** while the observed label was stable. That is a useful failure because the model found a plausible risk pattern without matching the observed decline label.

The next cell surfaces three false positives and three false negatives from the newly recomputed client-grouped holdout. Only pseudonymous content IDs are shown; no client names or URLs are exposed.


In [4]:
held = frame.iloc[te_g][[
    "content_id","is_declining_label","impressions_90d","sessions_90d",
    "avg_position","ctr","content_age_days","days_since_last_update",
    "word_count","trend_direction"
]].copy()
held["model_probability"] = group_probs
held["error_type"] = np.select([
    (held.is_declining_label==0)&(held.model_probability>=.5),
    (held.is_declining_label==1)&(held.model_probability<.5)
],["false_positive","false_negative"],default="correct")

print("Highest-confidence false positives:")
print(held[held.error_type=="false_positive"].sort_values("model_probability",ascending=False).head(3).to_string(index=False))
print("\nLowest-scored false negatives:")
print(held[held.error_type=="false_negative"].sort_values("model_probability").head(3).to_string(index=False))


Highest-confidence false positives:
          content_id  is_declining_label  impressions_90d  sessions_90d  avg_position  ctr  content_age_days  days_since_last_update  word_count trend_direction  model_probability     error_type
content_331182ca4cae                   0             3026            24          35.9  0.0               134                      20      3546.0              up           0.755029 false_positive
content_db1cd41b4b4f                   0             1482            19          12.9  0.0               105                     105      2221.0              up           0.740876 false_positive
content_f5013794ba57                   0              881             2          15.7  0.0               175                      20      3622.0             new           0.734883 false_positive

Lowest-scored false negatives:
          content_id  is_declining_label  impressions_90d  sessions_90d  avg_position  ctr  content_age_days  days_since_last_update  word_count trend_d

## 4. Claim rewrite

### Too-strong version
> “The Random Forest identifies pages that should be refreshed and will improve traffic.”

### Audited version
> **Observed and measured:** On the client-grouped holdout, the Random Forest ranks pages by the observed decline label at a measured Precision@50. The score is useful as **decision-support** for prioritizing manual review, but this audit does not establish that refreshing a flagged page will improve traffic.

A causal refresh claim would require an intervention or credible matched/control design with the refresh decision established before the outcome window.

**Words used throughout:** observed, measured, directional, decision-support. Avoid: proves, causes, will increase, guaranteed, predicts Google.


In [5]:
claim_check = {
    "observed": True,
    "measured": True,
    "directional": True,
    "decision_support": True,
    "causal_claim": False
}
print("Claim-language self-check:", claim_check)


Claim-language self-check: {'observed': True, 'measured': True, 'directional': True, 'decision_support': True, 'causal_claim': False}


## Self-check

- [x] Two paper findings + constructive methodology questions
- [x] Random row split vs client-grouped split
- [x] Base rate printed with metrics
- [x] Leakage audit covers label, decision, ID, and time-window risks
- [x] Real held-out failure examples
- [x] Safe claim language
- [x] No client names, URLs, or private queries

**Execution note:** the notebook is fully prepared, but a Jupyter kernel must run it before the final commit can truthfully be called “executed”. Open in Colab and use **Runtime → Run all**, then save the executed notebook back to this path.
